# S3, Domain randomization, two arms, held-out physics


Does randomizing training physics improve performance on physics the policy
never saw?

Two arms, identical in every respect except randomization: a **DR** arm that
samples cube mass, friction, actuator gain, joint damping, observation noise
and integer-frame action latency per episode, and a **baseline** arm with all
of it pinned to nominal. Both are then evaluated on a held-out band that is
disjoint from the training band **by construction**, defined in
`randomize.py` before either arm trained.

This is sim-to-sim transfer, not sim-to-real. There is no physical LEAP hand
here and calling it sim-to-real would be a lie. The methodology is the point
either way, and most published sim-to-real results answer a weaker version of
this question.

**Run S2 first.** Not because S3 needs its weights, the arms train from
scratch, but because S2 tells you whether the task is learnable at all and
what throughput to expect. Two arms is two training budgets.


---

### Before you run anything

1. **Runtime → Change runtime type → T4 GPU.** Every cell below assumes it.
2. **Keep this tab visible.** Free Colab disconnects an idle notebook after
   about 90 minutes and reclaims the runtime; `/content` does not survive it.
3. **The free tier has a quota you cannot see.** It is not published, it
   varies, and it is consumed by wall-clock GPU time whether or not you are
   computing. Expect a few hours a day, and expect to be cut off mid-run
   without warning. Every long-running cell here is written to survive that.

Checkpoints go to Google Drive, not to `/content`. That is the whole reason
the Drive cell exists, a run that checkpoints only to local disk loses
everything the moment the runtime is reclaimed, which on free Colab is the
normal way a session ends rather than an exceptional one.

## 1. Hardware, packages, Drive, code

In [ ]:
# --- what hardware did we actually get? ---
import subprocess, sys
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv'], capture_output=True, text=True).stdout)

# A label is not hardware. A Kaggle session advertised as "T4 x2" reported a
# P100 to the driver, which has compute capability 6.0 and cannot run several
# things a T4 can. Record what the driver says and quote it as such; never
# write "measured on a T4" because the runtime menu said T4.

In [ ]:
# --- packages ---
# jax with CUDA is preinstalled on Colab GPU runtimes. mujoco-mjx is not, and
# installing it can drag in a CPU-only jax wheel that silently replaces the
# working one. So: install, then re-check the device, and only reinstall jax
# if the check fails.
import subprocess, sys

def sh(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True)
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

sh(f'{sys.executable} -m pip install -q mujoco mujoco-mjx optax')

import importlib, jax
importlib.reload(jax)
if not any(d.platform == 'gpu' for d in jax.devices()):
    print('jax lost the GPU during install; reinstalling the CUDA wheel')
    sh(f'{sys.executable} -m pip install -q -U "jax[cuda12]"')
    raise SystemExit(
        'Reinstalled jax. Runtime -> Restart session, then run this cell '
        'again. (A restart is required: the CPU-only jax is already imported '
        'into this process and reimporting will not replace it.)')

In [ ]:
import jax, mujoco
print('jax     ', jax.__version__)
print('mujoco  ', mujoco.__version__)
print('devices ', jax.devices())
print('kind    ', getattr(jax.devices()[0], 'device_kind', '?'), '(as reported by the driver)')

# Hard stop, not a warning. On CPU a single mjx.step of the LEAP scene costs
# about 4 seconds and the compile runs past half an hour: a CPU session is not
# a slow run, it is no run.
assert any(d.platform == 'gpu' for d in jax.devices()), \
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then restart and re-run.'
print()
print('GPU OK')

In [ ]:
# --- Drive, for anything that must outlive this runtime ---
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/robotics-rl-portfolio')
DRIVE.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
print('drive :', DRIVE)
print('local :', WORK)

In [ ]:
# --- code ---
import os, shutil, subprocess, sys
from pathlib import Path

SRC = WORK / 'robotics-rl-portfolio'
REPO = 'https://github.com/JacobEGarcia/robotics-rl-portfolio.git'

def sh(cmd, **kw):
    print('$', cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=True, **kw)

if SRC.exists():
    sh(f'cd {SRC} && git pull --ff-only')
else:
    sh(f'git clone --depth 1 {REPO} {SRC}')

need = SRC / 's3_domain_rand/train.py'
if not need.exists():
    # Fallback for code that is committed locally but not pushed. Tar the
    # repo on your machine, drop it in Drive, and this picks it up:
    #   tar czf portfolio.tgz --exclude=assets --exclude=runs .
    tgz = DRIVE / 'portfolio.tgz'
    if tgz.exists():
        print(f'{need} missing from the clone; unpacking {tgz} over it')
        sh(f'tar xzf {tgz} -C {SRC}')
    if not need.exists():
        raise SystemExit(
            f'{need} is not in the cloned repo and no {tgz} was found.\n'
            f'Push it from your machine:\n'
            f'    cd ~/Downloads/hermestes/robotics-rl-portfolio && git push origin main\n'
            f'or upload a tarball to {tgz}.')

os.environ['PYTHONPATH'] = str(SRC)
sys.path.insert(0, str(SRC))
print()
print('code OK at', SRC)

In [ ]:
# --- the robot models ---
# assets/menagerie is gitignored in the portfolio repo on purpose: Menagerie
# is 2.3 GB of third-party assets and is itself a git repo, so it is fetched
# rather than vendored. A sparse checkout gets what is needed in a few
# seconds instead of pulling all of it.
MENAGERIE = SRC / 'assets' / 'menagerie'
WANT = ['leap_hand', 'franka_emika_panda', 'unitree_z1',
        'unitree_go1', 'unitree_h1', 'shadow_hand', 'dynamixel_2r']

if not (MENAGERIE / 'leap_hand' / 'right_hand.xml').exists():
    shutil.rmtree(MENAGERIE, ignore_errors=True)
    MENAGERIE.parent.mkdir(parents=True, exist_ok=True)
    sh('git clone --depth 1 --filter=blob:none --sparse '
       'https://github.com/google-deepmind/mujoco_menagerie.git ' + str(MENAGERIE))
    sh(f'cd {MENAGERIE} && git sparse-checkout set ' + ' '.join(WANT))

missing = [w for w in WANT if not (MENAGERIE / w).exists()]
assert not missing, f'sparse checkout did not produce: {missing}'
print('models OK:', sorted(p.name for p in MENAGERIE.iterdir() if p.is_dir())[:12])

## 2. Check the split before spending anything on it

The whole claim rests on the held-out band being outside the training band. That is enforced by construction in `randomize.py`, and verified here against actual draws, because "by construction" is a claim about code and this is the measurement.

In [ ]:
import jax, numpy as np
from s2_inhand.env import InHandConfig
from s3_domain_rand.env import DomainRandEnv
from s3_domain_rand.randomize import DomainRandConfig

env = DomainRandEnv(InHandConfig(), DomainRandConfig(), enabled=True)
keys = jax.random.split(jax.random.PRNGKey(0), 4096)
tr = env.sample_params(keys, held_out=False)
te = env.sample_params(keys, held_out=True)

cfg = env.dr
print(f"{'parameter':<16}{'train band':>22}{'held-out draws':>26}{'disjoint':>10}")
ok = True
for name, rng in (('cube_mass', cfg.cube_mass), ('cube_friction', cfg.cube_friction),
                  ('actuator_gain', cfg.actuator_gain), ('joint_damping', cfg.joint_damping)):
    a, b = np.asarray(tr[name]), np.asarray(te[name])
    clean = bool(((b < rng.train_lo) | (b > rng.train_hi)).all())
    ok &= clean
    print(f'{name:<16}[{rng.train_lo:.2f}, {rng.train_hi:.2f}]'.ljust(38)
          + f'[{b.min():.3f}, {b.max():.3f}]'.rjust(26) + f'{str(clean):>10}')

lat_tr, lat_te = np.asarray(tr['latency']), np.asarray(te['latency'])
clean = bool(lat_te.min() > lat_tr.max()); ok &= clean
print(f"{'latency':<16}[{lat_tr.min()}, {lat_tr.max()}] frames".ljust(38)
      + f'[{lat_te.min()}, {lat_te.max()}]'.rjust(26) + f'{str(clean):>10}')

assert ok, 'held-out draws landed inside the training band'
print('\nheld-out set is disjoint from training on every parameter')

### The batched model must not replicate the whole model

`jax.vmap` adds a leading axis to every leaf of whatever it returns, so vmapping a function that returns an MJX model replicates mesh vertices, convex hulls and body trees once per environment. At the batch sizes MJX is worth using at, that is an out-of-memory error whose message says nothing about domain randomization. Only four arrays should carry a batch axis.

In [ ]:
import jax.tree_util as tu
mb, axes = env.batched_model(tr)
n_batched = sum(1 for l in tu.tree_leaves(axes) if l == 0)
shared = sum(np.prod(l.shape) * l.dtype.itemsize
             for l in tu.tree_leaves(env.base.model) if hasattr(l, 'shape'))
total = sum(np.prod(l.shape) * l.dtype.itemsize
            for l in tu.tree_leaves(mb) if hasattr(l, 'shape'))
print(f'leaves carrying a batch axis : {n_batched} (expected 4)')
print(f'model bytes, shared          : {shared/1e6:.2f} MB')
print(f'model bytes, batched x4096   : {total/1e6:.2f} MB  ({total/shared:.1f}x)')
assert n_batched == 4, 'more than the four randomized fields got batched'
print('\nbatching OK')

## 3. Train both arms

Same file, same optimiser, same seed handling, same curriculum. The curriculum is a fixed function of step count rather than of success rate: under S2's adaptive rule the DR arm, solving a harder problem, advances more slowly, so at any given step the two arms would be training on *different task distributions* and the final comparison would silently mix "DR generalises better" with "one arm spent longer on easy targets".

Set `ARM` and run. Each arm needs its own budget; expect to come back to this cell across several sessions.

In [ ]:
ARM         = 'dr'          # 'dr' or 'baseline' -- run both
TOTAL_STEPS = 30_000_000
NUM_ENVS    = 2048          # use the knee S2's section 3 measured
MAX_HOURS   = 2.5

import os, subprocess, sys
os.chdir(SRC)
CKPT_LOCAL = WORK / 's3' / ARM
CKPT_DRIVE = DRIVE / 's3_checkpoints' / ARM

cmd = [sys.executable, '-u', '-m', 's3_domain_rand.train',
       '--arm', ARM,
       '--total-steps', str(TOTAL_STEPS),
       '--num-envs', str(NUM_ENVS),
       '--ckpt-dir', str(CKPT_LOCAL),
       '--ckpt-mirror', str(CKPT_DRIVE),
       '--max-hours', str(MAX_HOURS),
       '--resume']
print('$', ' '.join(cmd), flush=True)

p = subprocess.Popen(cmd, env=dict(os.environ, PYTHONPATH=str(SRC)),
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
try:
    for line in p.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    p.terminate(); print('\ninterrupted; newest checkpoint is in Drive')
p.wait()

## 4. Evaluate, once both arms exist

Three conditions, not two: nominal physics, in-distribution physics, held-out physics. Without the first two, a DR arm that is simply worse everywhere and a DR arm that trades in-distribution performance for robustness look identical, and that trade is the interesting result.

Both arms see the identical held-out draws, from a seed unrelated to either training seed. The headline is the **paired** difference: the arms are scored on the same episodes, and pairing removes the episode-difficulty variance that dominates the unpaired numbers.

In [ ]:
DR_DIR   = DRIVE / 's3_checkpoints' / 'dr'
BASE_DIR = DRIVE / 's3_checkpoints' / 'baseline'
for d in (DR_DIR, BASE_DIR):
    assert list(d.glob('ckpt_*.pkl')), f'no checkpoints in {d}; train that arm first'

r = subprocess.run([sys.executable, '-u', '-m', 's3_domain_rand.eval',
                    '--dr', str(DR_DIR), '--baseline', str(BASE_DIR),
                    '--episodes', '512',
                    '--out', str(DRIVE / 's3_eval.json')],
                   env=dict(os.environ, PYTHONPATH=str(SRC)))
print('exit', r.returncode)

In [ ]:
import json
res = json.loads((DRIVE / 's3_eval.json').read_text())
print(f"{'condition':<12}{'DR':>18}{'baseline':>18}{'paired DR - base':>26}")
for kind, row in res['conditions'].items():
    p = row['paired_dr_minus_baseline']
    print(f"{kind:<12}{row['dr']['success_rate']:>17.1%}"
          f"{row['baseline']['success_rate']:>18.1%}"
          f"{p['point']:>+16.3f} [{p['lo']:+.3f}, {p['hi']:+.3f}]")
print()
print('An interval that straddles zero is a null result. Report it as one.')